# KMC Permeation Simulation

BKL kinetic Monte Carlo on a 2D dual-layer alloy grid, followed by Fick's law
flux evaluation and Sieverts' law diagnosis.

**Inputs** (from `tst_calculation.ipynb`):
- `rate_dict_T{T}K.json` — rate constants at target temperature
- Bulk diffusivity D(T) from `models/diffusivity_workflow.py` MSD results.

**Outputs**:
- θ(t) and C(t) steady-state plots
- J vs √P Sieverts plot
- Bottleneck diagnosis (R²)

---

**Grid events**

| Kind | Rate |
|---|---|
| H₂ adsorption | R_strike(P,T) × k_diss(pair) |
| H₂ desorption | k_des(pair) |
| H* surface diffusion | k_surf_diff(pair) |
| H* → H subsurface (Hop A) | k_entry(elem) |
| H subsurface → H* (Hop A rev) | k_exit(elem) |
| H subsurface → bulk drain | k_drain = D/dx² |


## Cell 1 — Imports & configuration

In [4]:
import json
import os
import sys

import numpy as np

# Add parent directory to path
parent_dir = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from models.config import BASE_DIR
from models.permeation import arrhenius_diffusivity

WORK_DIR    = os.path.join(BASE_DIR, 'calculation')
RESULTS_DIR = os.path.join(WORK_DIR, 'results')

# ── Temperature & material parameters ─────────────────────────────────────────
T_K    = 700.0          # K
A0_M   = 3.52e-10       # m  — Ni lattice constant
L_M    = 1e-3           # m  — membrane thickness (1 mm)

# ── Bulk diffusivity from LAMMPS MSD (diffusivity_workflow.py) ────────────────
DIFF_JSON = os.path.join(RESULTS_DIR, 'diffusivity_arrhenius.json')
if os.path.exists(DIFF_JSON):
    with open(DIFF_JSON) as f:
        _diff = json.load(f)
    D0_M2S = _diff['D0_m2s']
    E_D_EV = _diff['E_D_eV']
    D_M2S  = arrhenius_diffusivity(D0_M2S, E_D_EV, T_K)
    print(f'D({T_K} K) = {D_M2S:.3e} m²/s  [from Arrhenius fit]')
else:
    D_M2S = 1e-9
    print(f'WARNING: {DIFF_JSON} not found. Using placeholder D_M2S = {D_M2S}')

# ── KMC grid ──────────────────────────────────────────────────────────────────
NX, NY = 20, 20         # grid dimensions
SEED   = 42

# ── Pressure sweep ────────────────────────────────────────────────────────────
P_VALS_PA = [1e3, 1e4, 1e5, 5e5, 1e6]   # Pa

# ── Rate dict JSON ────────────────────────────────────────────────────────────
RATE_DICT_JSON = os.path.join(RESULTS_DIR, f'rate_dict_T{int(T_K)}K.json')

print(f'T_K        = {T_K} K')
print(f'D_M2S      = {D_M2S} m²/s')
print(f'L_M        = {L_M} m')
print(f'Grid       = {NX}×{NY}')
print(f'RATE_DICT  : {RATE_DICT_JSON}')

T_K        = 700.0 K
D_M2S      = 1e-09 m²/s
L_M        = 0.001 m
Grid       = 20×20
RATE_DICT  : /projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel/calculation/results/rate_dict_T700K.json


## Cell 2 — Load rate dict and build KMC rate_dict

The `tst_calculation.ipynb` emits `k_forward` / `k_reverse` per NEB label.
Here we map them to the KMC format:

- `k_diss[pair]`  → dimensionless Boltzmann sticking = `exp(−Ea/kT)` (multiplied by R_strike inside KMC)
- `k_des[pair]`   → full TST rate [s⁻¹] (k_reverse from Hop A dissociation NEB)
- `k_entry[elem]` → Hop A k_forward averaged over matching labels [s⁻¹]
- `k_exit[elem]`  → Hop A k_reverse [s⁻¹]

**Adjust the mapping below** to match your NEB label convention.

In [5]:
from models.tst_rates import BOLTZMANN_eV
from models.parsers import parse_barrier_file

with open(RATE_DICT_JSON) as f:
    tst_rates = json.load(f)

# ── k_entry / k_exit from Hop A NEB labels ────────────────────────────────────
k_diss   = {}
k_des    = {}
k_entry  = {}
k_exit   = {}

for label, r in tst_rates.items():
    if label.startswith('hopa_'):
        sid = label[len('hopa_'):]
        for elem in ('Ni', 'Mo', 'Cr', 'Fe'):
            if elem in sid:
                k_entry[elem] = r['k_forward']
                k_exit[elem]  = r['k_reverse']
                break

# ── k_diss / k_des from H₂ dissociation NEB ──────────────────────────────────
kB_T = BOLTZMANN_eV * T_K
NU_DISS = 1e13   # s⁻¹ — attempt frequency for desorption

DISS_VIB_JSON  = os.path.join(WORK_DIR, 'neb', 'diss_vib_rates.json')
DISS_JOBS_JSON = os.path.join(WORK_DIR, 'neb', 'diss_jobs.json')

if os.path.exists(DISS_VIB_JSON):
    # Phase E ZPE-corrected rates (preferred)
    with open(DISS_VIB_JSON) as f:
        diss_vib = json.load(f)
    for lbl, dv in diss_vib.items():
        pair = tuple(dv['pair'])
        k_diss[pair] = dv['nu'] * np.exp(-dv['Ea_zpe'] / kB_T)
        k_des[pair]  = dv['nu'] * np.exp(-dv['Ed_zpe'] / kB_T)
    print(f'Loaded {len(k_diss)} k_diss/k_des pairs from diss_vib_rates.json (ZPE-corrected)')
elif os.path.exists(DISS_JOBS_JSON):
    with open(DISS_JOBS_JSON) as f:
        diss_jobs = json.load(f)
    for job in diss_jobs:
        bf = job.get('barrier_file', '')
        if not os.path.exists(bf):
            continue
        bd   = parse_barrier_file(bf)
        pair = tuple(sorted(job['sid'].replace('-', '').split('_')[:2]))
        k_diss[pair] = np.exp(-bd['E_abs'] / kB_T)
        k_des[pair]  = NU_DISS * np.exp(-bd['E_des'] / kB_T)
    print(f'Loaded {len(k_diss)} k_diss/k_des pairs from diss_jobs.json (raw barriers)')
else:
    print(f'WARNING: neither diss_vib_rates.json nor diss_jobs.json found — using placeholders.')
    for pair in [('Ni', 'Ni'), ('Mo', 'Ni'), ('Cr', 'Ni'), ('Fe', 'Ni'),
                 ('Mo', 'Mo'), ('Cr', 'Mo'), ('Fe', 'Mo'),
                 ('Cr', 'Cr'), ('Fe', 'Cr'), ('Fe', 'Fe')]:
        k_diss[pair] = np.exp(-0.5  / kB_T)
        k_des[pair]  = NU_DISS * np.exp(-1.2 / kB_T)

kmc_rate_dict = {
    'k_diss':  k_diss,
    'k_des':   k_des,
    'k_entry': k_entry,
    'k_exit':  k_exit,
    # 'k_surf_diff': {}   # add if surface diffusion NEB is available
}

print(f'k_entry: {k_entry}')
print(f'k_exit : {k_exit}')
print(f'k_diss pairs: {list(k_diss.keys())[:3]} ...')

FileNotFoundError: [Errno 2] No such file or directory: '/projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel/calculation/results/rate_dict_T700K.json'

## Cell 3 — Single-pressure KMC run: θ(t) and C(t)

Run at one representative pressure to confirm steady-state convergence.

In [ ]:
import matplotlib.pyplot as plt
from models.kmc import make_grid, run_kmc, surface_coverage, subsurface_concentration

np.random.seed(SEED)
P_single = 1e5   # Pa

grid = make_grid(NX, NY, seed=SEED)
out  = run_kmc(grid, kmc_rate_dict, P_single, T_K, D_M2S, A0_M, n_steps=20_000)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(out['t_arr'], out['theta_arr'], lw=1.0)
axes[0].set_xlabel('Time  [s]')
axes[0].set_ylabel('Surface coverage θ')
axes[0].set_title(f'P = {P_single:.0e} Pa')

axes[1].plot(out['t_arr'], out['n_sub_arr'], lw=1.0, color='coral')
axes[1].set_xlabel('Time  [s]')
axes[1].set_ylabel('Subsurface H count')
axes[1].set_title('Subsurface population')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'kmc_trajectory.png'), dpi=150)
plt.show()

C0_final = subsurface_concentration(grid, A0_M)
print(f'Final θ  = {surface_coverage(grid):.3f}')
print(f'Final C0 = {C0_final:.3e} atoms/m³')

## Cell 4 — Pressure sweep → collect J values

In [ ]:
from models.permeation import sweep_pressure

np.random.seed(SEED)
print('Sweeping pressures ...')

sweep = sweep_pressure(
    P_vals_Pa  = P_VALS_PA,
    rate_dict  = kmc_rate_dict,
    D_m2s      = D_M2S,
    L_m        = L_M,
    T_K        = T_K,
    a0_m       = A0_M,
    nx=NX, ny=NY,
    seed=SEED,
    kmc_kwargs = {'window': 2000, 'rtol': 0.02, 'max_steps': 500_000},
)

print(f'\nResults:')
for P, J, C0, conv in zip(sweep['P_vals'], sweep['J_vals'],
                           sweep['C0_vals'], sweep['converged']):
    print(f'  P={P:.2e} Pa  C0={C0:.3e}  J={J:.3e} atoms/m²/s  conv={conv}')

## Cell 5 — Plot J vs √P and Sieverts' law fit

In [ ]:
from models.permeation import check_sieverts_law

result = check_sieverts_law(
    P_vals_Pa = sweep['P_vals'],
    J_vals    = sweep['J_vals'],
    plot      = True,
)

print(f"\nSlope     : {result['slope']:.4e}")
print(f"Intercept : {result['intercept']:.4e}")
print(f"R²        : {result['r_squared']:.4f}")
print(f"Sieverts? : {result['is_sieverts']}")

if result['is_sieverts']:
    print('\n→ Transport is BULK-DIFFUSION limited (Sieverts law holds).')
    print('  Increasing membrane thickness will reduce flux proportionally.')
else:
    print('\n→ Transport is SURFACE / ENTRY KINETICS limited.')
    print('  Reducing entry barrier or modifying surface composition will improve flux.')

## Cell 6 — Save sweep results

In [ ]:
sweep_out = os.path.join(RESULTS_DIR, f'permeation_sweep_T{int(T_K)}K.json')
with open(sweep_out, 'w') as f:
    json.dump({'sweep': sweep, 'sieverts': result, 'T_K': T_K,
               'D_m2s': D_M2S, 'L_m': L_M}, f, indent=2)
print(f'Saved: {sweep_out}')

## Cell 7 — Richardson-Sieverts permeability: all three S₀ options

Φ(T) = D(T) × S(T)   →   J = Φ (√P_high − √P_low) / L

**ΔH_sol** = ΔH_diss / 2 + ΔH_entry (Hop A reaction energy, from NEB)

| Option | S₀ source | Description |
|---|---|---|
| 1 | Lattice-site density | S₀ = 4/a₀³  (geometric upper bound) |
| 2 | TST rates (detailed balance) | S(T) from k_diss, k_des, k_entry, k_exit |
| 3 | KMC empirical | S = C₀/√P fitted over the pressure sweep |

In [ ]:
from models.permeation import (
    arrhenius_diffusivity,
    lattice_site_S0,
    solubility_from_rates,
    fit_solubility_from_kmc,
    sieverts_solubility,
    permeability,
    richardson_flux,
)

# ── Material parameters ────────────────────────────────────────────────────────
# Use D₀ / E_D loaded in Cell 1 from diffusivity_arrhenius.json if available;
# otherwise fall back to typical Ni values.
if 'D0_M2S' not in globals() or 'E_D_EV' not in globals():
    D0_M2S = 1.5e-7   # m²/s  — typical for Ni
    E_D_EV  = 0.40    # eV

# ΔH_sol = ΔH_diss / 2 + ΔH_entry
# ΔH_diss from dissociation NEB delta_E; ΔH_entry from Hop A delta_E.
DH_DISS_EV  = -0.50   # eV  (exothermic dissociation, sign: IS → FS)
DH_ENTRY_EV = +0.20   # eV  (endothermic sub-surface entry)
DH_SOL_EV   = DH_DISS_EV / 2.0 + DH_ENTRY_EV

# Membrane parameters
P_HIGH_PA = 1e5   # Pa   — feed side
P_LOW_PA  = 0.0   # Pa   — permeate side (fully swept)
L_M       = 1e-3  # m    — membrane thickness (1 mm)

D_T = arrhenius_diffusivity(D0_M2S, E_D_EV, T_K)
print(f'D({T_K} K) = {D_T:.3e} m²/s')
print(f'ΔH_sol    = {DH_SOL_EV:.3f} eV  ({DH_SOL_EV*1000:.1f} meV)')

# ── Option 1: Lattice-site density ────────────────────────────────────────────
S0_lat = lattice_site_S0(A0_M)
S1     = sieverts_solubility(DH_SOL_EV, S0_lat, T_K)
Phi1   = permeability(D_T, S1)
J1     = richardson_flux(Phi1, P_HIGH_PA, P_LOW_PA, L_M)
print(f'\nOption 1 (lattice site density):')
print(f'  S₀      = {S0_lat:.3e} atoms/m³/Pa^½')
print(f'  S({T_K} K) = {S1:.3e} atoms/m³/Pa^½')
print(f'  Φ       = {Phi1:.3e} atoms·m⁻¹·s⁻¹·Pa^(-½)')
print(f'  J       = {J1:.3e} atoms/m²/s  at P={P_HIGH_PA:.0e} Pa')

# ── Option 2: TST rates (detailed balance) ────────────────────────────────────
repr_entry = list(kmc_rate_dict['k_entry'].values())[0] if kmc_rate_dict['k_entry'] else 1e9
repr_exit  = list(kmc_rate_dict['k_exit'].values())[0]  if kmc_rate_dict['k_exit']  else 1e8
repr_diss  = list(kmc_rate_dict['k_diss'].values())[0]  if kmc_rate_dict['k_diss']  else 0.01
repr_des   = list(kmc_rate_dict['k_des'].values())[0]   if kmc_rate_dict['k_des']   else 1e11

S2   = solubility_from_rates(repr_diss, repr_des, repr_entry, repr_exit, A0_M, T_K)
Phi2 = permeability(D_T, S2)
J2   = richardson_flux(Phi2, P_HIGH_PA, P_LOW_PA, L_M)
print(f'\nOption 2 (detailed balance from TST rates):')
print(f'  S({T_K} K) = {S2:.3e} atoms/m³/Pa^½')
print(f'  Φ       = {Phi2:.3e} atoms·m⁻¹·s⁻¹·Pa^(-½)')
print(f'  J       = {J2:.3e} atoms/m²/s  at P={P_HIGH_PA:.0e} Pa')

# ── Option 3: KMC empirical fit ────────────────────────────────────────────────
kmc_sol = fit_solubility_from_kmc(sweep)
S3   = kmc_sol['S_mean']
Phi3 = permeability(D_T, S3)
J3   = richardson_flux(Phi3, P_HIGH_PA, P_LOW_PA, L_M)
print(f'\nOption 3 (KMC empirical, {kmc_sol["n_converged"]} converged points):')
print(f'  S̄({T_K} K)  = {S3:.3e} ± {kmc_sol["S_std"]:.1e} atoms/m³/Pa^½')
print(f'  Φ        = {Phi3:.3e} atoms·m⁻¹·s⁻¹·Pa^(-½)')
print(f'  J        = {J3:.3e} atoms/m²/s  at P={P_HIGH_PA:.0e} Pa')

# ── Consistency check: Fick ↔ Richardson ──────────────────────────────────────
J_fick_at_P = D_T * S3 * np.sqrt(P_HIGH_PA) / L_M
print(f'\nConsistency: J_Fick(S_KMC)={J_fick_at_P:.3e}  vs  J_Richardson(S_KMC)={J3:.3e}')

## Cell 8 — Φ(T) Arrhenius plot: D(T) × S(T)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

T_range = np.linspace(400, 1100, 300)   # K

# Pre-compute S values for all three options at each T
D_arr   = np.array([arrhenius_diffusivity(D0_M2S, E_D_EV, T) for T in T_range])
S1_arr  = np.array([sieverts_solubility(DH_SOL_EV, S0_lat, T) for T in T_range])
S2_arr  = np.array([solubility_from_rates(repr_diss, repr_des,
                     repr_entry, repr_exit, A0_M, T) for T in T_range])
# Option 3: S₀ from KMC at single T — extrapolate Arrhenius with same ΔH_sol
S3_arr  = np.array([sieverts_solubility(DH_SOL_EV, S3, T) for T in T_range])

Phi1_arr = D_arr * S1_arr
Phi2_arr = D_arr * S2_arr
Phi3_arr = D_arr * S3_arr

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: Φ vs T (linear) ─────────────────────────────────────────────────────
ax = axes[0]
ax.plot(T_range, Phi1_arr, label='Option 1 (lattice)',    color='steelblue', lw=1.6)
ax.plot(T_range, Phi2_arr, label='Option 2 (rates)',      color='coral',     lw=1.6)
ax.plot(T_range, Phi3_arr, label='Option 3 (KMC fit)',    color='seagreen',  lw=1.6, ls='--')
ax.axvline(T_K, color='k', ls=':', lw=1.0, label=f'T={T_K:.0f} K')
ax.set_xlabel('Temperature  [K]')
ax.set_ylabel('Φ  [atoms·m⁻¹·s⁻¹·Pa^(−½)]')
ax.set_title('Permeability Φ(T) = D(T) × S(T)')
ax.legend(fontsize=8)

# ── Right: Arrhenius plot — ln(Φ) vs 1000/T ───────────────────────────────────
ax2 = axes[1]
inv_T = 1000.0 / T_range   # 1000/T for readability
ax2.plot(inv_T, np.log10(Phi1_arr), label='Option 1', color='steelblue', lw=1.6)
ax2.plot(inv_T, np.log10(Phi2_arr), label='Option 2', color='coral',     lw=1.6)
ax2.plot(inv_T, np.log10(Phi3_arr), label='Option 3', color='seagreen',  lw=1.6, ls='--')
ax2.axvline(1000.0 / T_K, color='k', ls=':', lw=1.0, label=f'T={T_K:.0f} K')
ax2.set_xlabel('1000 / T  [K⁻¹]')
ax2.set_ylabel('log₁₀(Φ)')
ax2.set_title('Arrhenius plot of Φ(T)')
ax2.legend(fontsize=8)
ax2.invert_xaxis()

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'permeability_vs_T.png'), dpi=150)
plt.show()

# ── Effective activation energy: E_Phi = E_D + ΔH_sol ────────────────────────
# From Arrhenius: Φ = Φ₀ exp(-(E_D + ΔH_sol) / kBT)
E_phi = E_D_EV + DH_SOL_EV
print(f'Effective permeation activation energy E_Φ = E_D + ΔH_sol')
print(f'  = {E_D_EV:.2f} + {DH_SOL_EV:.3f} = {E_phi:.3f} eV')

# Save permeability results
perm_out = os.path.join(RESULTS_DIR, f'permeability_T{int(T_K)}K.json')
with open(perm_out, 'w') as f:
    import json as _json
    _json.dump({
        'T_K': T_K,
        'D0_m2s': D0_M2S, 'E_D_eV': E_D_EV,
        'dH_sol_eV': DH_SOL_EV,
        'option1': {'S': S1, 'Phi': Phi1, 'J': J1},
        'option2': {'S': S2, 'Phi': Phi2, 'J': J2},
        'option3': {'S': S3, 'Phi': Phi3, 'J': J3,
                    'S_std': kmc_sol['S_std'], 'n_converged': kmc_sol['n_converged']},
        'P_high_Pa': P_HIGH_PA, 'P_low_Pa': P_LOW_PA, 'L_m': L_M,
        'E_phi_eV': E_phi,
    }, f, indent=2)
print(f'Saved: {perm_out}')

## Cell 9 — Multi-temperature S₀ Arrhenius fit (Option 3)

Sweeps KMC at several temperatures, collects S̄(T) = C₀/√P from each sweep,
then fits ln(S) vs 1/T → Arrhenius S₀ and ΔH_sol for Option 3.

**Requires** `rate_dict_T{T}K.json` for each T in `T_RANGE_K`.  
Run `tst_calculation.ipynb` at each temperature first, or set `T_RANGE_K = [T_K]`
for a single-temperature diagnostic.

**Outputs**: `solubility_arrhenius_kmc.json`, `solubility_arrhenius_kmc.png`

In [ ]:
KB_EV = 8.617333262e-5   # eV/K

T_RANGE_K = [500, 600, 700, 800, 900]   # K — must have rate_dict_T{T}K.json for each

S_arr, T_arr = [], []
print('Multi-T KMC sweep for Arrhenius S₀ (Option 3)')
print('─' * 55)

for T_i in T_RANGE_K:
    json_i = os.path.join(RESULTS_DIR, f'rate_dict_T{int(T_i)}K.json')
    if not os.path.exists(json_i):
        print(f'  T={T_i:4.0f} K  ← rate_dict JSON missing, skipping.')
        continue

    with open(json_i) as f:
        tst_i = json.load(f)

    kB_Ti = KB_EV * T_i
    k_e_i, k_x_i, k_d_i, k_ds_i = {}, {}, {}, {}
    for label, r in tst_i.items():
        if label.startswith('hopa_'):
            sid = label[len('hopa_'):]
            for elem in ('Ni', 'Mo', 'Cr', 'Fe'):
                if elem in sid:
                    k_e_i[elem] = r['k_forward']
                    k_x_i[elem] = r['k_reverse']
                    break

    if os.path.exists(DISS_VIB_JSON):
        # Phase E ZPE-corrected rates (preferred)
        with open(DISS_VIB_JSON) as f_dv:
            dv_i = json.load(f_dv)
        for lbl, dv in dv_i.items():
            pair = tuple(dv['pair'])
            k_d_i[pair]  = dv['nu'] * np.exp(-dv['Ea_zpe'] / kB_Ti)
            k_ds_i[pair] = dv['nu'] * np.exp(-dv['Ed_zpe'] / kB_Ti)
    elif os.path.exists(DISS_JOBS_JSON):
        from models.parsers import parse_barrier_file as _pbf
        with open(DISS_JOBS_JSON) as f_dj:
            dj_i = json.load(f_dj)
        for job in dj_i:
            bf = job.get('barrier_file', '')
            if not os.path.exists(bf):
                continue
            bd   = _pbf(bf)
            pair = tuple(sorted(job['sid'].replace('-', '').split('_')[:2]))
            k_d_i[pair]  = np.exp(-bd['E_abs'] / kB_Ti)
            k_ds_i[pair] = 1e13 * np.exp(-bd['E_des'] / kB_Ti)
    else:
        for pair in [('Ni', 'Ni'), ('Mo', 'Ni'), ('Cr', 'Ni'), ('Fe', 'Ni'),
                     ('Mo', 'Mo'), ('Cr', 'Mo'), ('Fe', 'Mo'),
                     ('Cr', 'Cr'), ('Fe', 'Cr'), ('Fe', 'Fe')]:
            k_d_i[pair]  = np.exp(-0.5  / kB_Ti)
            k_ds_i[pair] = 1e13 * np.exp(-1.2 / kB_Ti)

    kmc_i = {'k_diss': k_d_i, 'k_des': k_ds_i, 'k_entry': k_e_i, 'k_exit': k_x_i}
    D_i   = arrhenius_diffusivity(D0_M2S, E_D_EV, T_i)
    np.random.seed(SEED)
    sw_i  = sweep_pressure(
        P_vals_Pa  = P_VALS_PA,
        rate_dict  = kmc_i,
        D_m2s      = D_i,
        L_m        = L_M,
        T_K        = T_i,
        a0_m       = A0_M,
        nx=NX, ny=NY, seed=SEED,
        kmc_kwargs = {'window': 2000, 'rtol': 0.02, 'max_steps': 500_000},
    )
    sol_i = fit_solubility_from_kmc(sw_i)
    if sol_i['n_converged'] == 0:
        print(f'  T={T_i:4.0f} K  ← no converged KMC points, skipping.')
        continue
    S_arr.append(sol_i['S_mean'])
    T_arr.append(float(T_i))
    print(f'  T={T_i:4.0f} K  S̄ = {sol_i["S_mean"]:.3e}  (n_conv={sol_i["n_converged"]})')

if len(S_arr) < 2:
    print('\nWARNING: fewer than 2 valid S values — cannot fit Arrhenius.')
    print('Run tst_calculation.ipynb at multiple temperatures and re-run this cell.')
else:
    S_arr = np.array(S_arr)
    T_arr = np.array(T_arr)

    slope, intercept = np.polyfit(1.0 / T_arr, np.log(S_arr), 1)
    dH_sol_kmc = -slope * KB_EV       # eV
    S0_kmc     = np.exp(intercept)    # atoms m⁻³ Pa^(−½)

    log_S_pred = slope / T_arr + intercept
    ss_res = np.sum((np.log(S_arr) - log_S_pred) ** 2)
    ss_tot = np.sum((np.log(S_arr) - np.mean(np.log(S_arr))) ** 2)
    r2_sol = 1.0 - ss_res / ss_tot if ss_tot > 0 else 1.0

    print(f'\nArrhenius fit  S(T) = S₀ exp(−ΔH_sol / kT)')
    print(f'  S₀       = {S0_kmc:.3e} atoms·m⁻³·Pa^(−½)')
    print(f'  ΔH_sol   = {dH_sol_kmc:.3f} eV')
    print(f'  R²       = {r2_sol:.4f}')

    T_plot  = np.linspace(T_arr.min() * 0.9, T_arr.max() * 1.1, 200)
    D_plot  = np.array([arrhenius_diffusivity(D0_M2S, E_D_EV, T) for T in T_plot])
    S_fit   = S0_kmc * np.exp(-dH_sol_kmc / (KB_EV * T_plot))
    Phi_fit = D_plot * S_fit
    D_pts   = np.array([arrhenius_diffusivity(D0_M2S, E_D_EV, T) for T in T_arr])

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ax0 = axes[0]
    ax0.plot(T_plot, Phi_fit, color='purple', lw=2.0, label='Arrhenius fit')
    ax0.scatter(T_arr, D_pts * S_arr, color='purple', zorder=5, label='KMC S̄(T)')
    ax0.set_xlabel('Temperature  [K]')
    ax0.set_ylabel('Φ  [atoms·m⁻¹·s⁻¹·Pa^(−½)]')
    ax0.set_title('Option 3 — Φ(T) from KMC Arrhenius S₀')
    ax0.legend(fontsize=8)

    ax1 = axes[1]
    ax1.plot(1000.0 / T_plot, np.log10(Phi_fit), color='purple', lw=2.0, label='Arrhenius fit')
    ax1.scatter(1000.0 / T_arr, np.log10(D_pts * S_arr),
                color='purple', zorder=5, label='KMC S̄(T)')
    ax1.set_xlabel('1000 / T  [K⁻¹]')
    ax1.set_ylabel('log₁₀(Φ)')
    ax1.set_title(f'ΔH_sol (KMC) = {dH_sol_kmc:.3f} eV   R² = {r2_sol:.3f}')
    ax1.invert_xaxis()
    ax1.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'solubility_arrhenius_kmc.png'), dpi=150)
    plt.show()

    sol_out = os.path.join(RESULTS_DIR, 'solubility_arrhenius_kmc.json')
    with open(sol_out, 'w') as f:
        json.dump({
            'T_K_arr':       T_arr.tolist(),
            'S_mean_arr':    S_arr.tolist(),
            'S0_kmc':        S0_kmc,
            'dH_sol_kmc_eV': dH_sol_kmc,
            'r2_fit':        r2_sol,
            'D0_m2s':        D0_M2S,
            'E_D_eV':        E_D_EV,
        }, f, indent=2)
    print(f'Saved: {sol_out}')